# Variational Autoencoders from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/variational_autoencoder_from_scratch.ipynb)

An autoencoder squeezes an image into a few numbers and rebuilds it. A variational autoencoder adds one term to the loss, and that term is the difference between a model that compresses and a model that can generate.

This notebook builds one in NumPy with the gradients derived by hand, then measures what the term buys: with it, sampling the prior reaches all ten digit classes; without it, three.

Everything runs on a CPU in about two minutes. No GPU, no framework.

Companion post: [Variational Autoencoders from Scratch](https://sesen.ai/blog/variational-autoencoder-from-scratch)

## 1. The data

scikit-learn's 8x8 digits, scaled to [0, 1] because the reconstruction term below treats every pixel as a Bernoulli probability.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression

X, y = load_digits(return_X_y=True)
X = X / 16.0                      # a Bernoulli likelihood needs the data in [0, 1]
print("data:", X.shape, "range", X.min(), X.max())

## 2. The model

Encoder to `(mu, logvar)`, a reparameterised sample, decoder to pixel logits. The loss is

```
L = reconstruction  +  beta * KL( q(z|x) || N(0, I) )
```

Setting `beta = 0` leaves a plain autoencoder, which is the control this notebook keeps coming back to.

Two things make the backward pass short enough to write out by hand. The Bernoulli likelihood's derivative collapses to `p - x`, because the sigmoid and the cross-entropy cancel. And the KL against a unit Gaussian has a closed form, so `d(KL)/d(mu) = mu` falls straight out.

In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-np.clip(x, -60, 60)))


class Dense:
    def __init__(self, n_in, n_out, rng, gain=None):
        self.W = rng.normal(0, gain or np.sqrt(2 / n_in), (n_in, n_out))
        self.b = np.zeros(n_out)

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, g):
        self.gW, self.gb = self.x.T @ g, g.sum(0)
        return g @ self.W.T


class Stack:                                  # Dense -> ReLU -> ... -> Dense
    def __init__(self, widths, rng):
        self.L = [Dense(a, b, rng) for a, b in zip(widths[:-1], widths[1:])]

    def forward(self, x):
        self.m = []
        for i, l in enumerate(self.L):
            x = l.forward(x)
            if i < len(self.L) - 1:
                self.m.append(x > 0)
                x = x * self.m[-1]
        return x

    def backward(self, g):
        for i in reversed(range(len(self.L))):
            if i < len(self.L) - 1:
                g = g * self.m[i]
            g = self.L[i].backward(g)
        return g

    def pg(self):
        return [(l.W, l.gW) for l in self.L] + [(l.b, l.gb) for l in self.L]

    def p(self):
        return [l.W for l in self.L] + [l.b for l in self.L]


class VAE:
    def __init__(self, n_in, hidden=32, latent=2, seed=0):
        rng = np.random.default_rng(seed)
        self.latent = latent
        self.enc = Stack([n_in, hidden, hidden], rng)
        self.mu_head = Dense(hidden, latent, rng, gain=0.05)
        self.lv_head = Dense(hidden, latent, rng, gain=0.05)
        self.dec = Stack([latent, hidden, hidden, n_in], rng)

    def forward(self, x, rng, sample=True):
        h = self.enc.forward(x)
        self.mu, self.lv = self.mu_head.forward(h), self.lv_head.forward(h)
        self.sd = np.exp(0.5 * np.clip(self.lv, -20, 20))
        # The reparameterisation trick: the noise is an input, so dz/dmu = 1 and
        # gradient reaches the encoder. Sampling inside the graph would cut it.
        self.eps = rng.standard_normal(self.mu.shape) if sample else 0.0
        self.z = self.mu + self.sd * self.eps
        return self.dec.forward(self.z)

    def loss(self, x, beta):
        self.p_out, self.x = sigmoid(self.dec.forward(self.z)), x
        recon = -(x * np.log(self.p_out + 1e-7)
                  + (1 - x) * np.log(1 - self.p_out + 1e-7)).sum(1)
        kl = -0.5 * (1 + self.lv - self.mu**2 - np.exp(np.clip(self.lv, -20, 20))).sum(1)
        return recon.mean(), kl.mean()

    def backward(self, beta):
        n = len(self.x)
        g_z = self.dec.backward((self.p_out - self.x) / n)     # d(recon)/d(out) = p - x
        g_mu = g_z + beta * self.mu / n                        # + d(KL)/d(mu)
        g_lv = (g_z * self.eps * 0.5 * self.sd
                + beta * 0.5 * (np.exp(np.clip(self.lv, -20, 20)) - 1) / n)
        self.enc.backward(self.mu_head.backward(g_mu) + self.lv_head.backward(g_lv))

    def pg(self):
        return (self.enc.pg() + [(self.mu_head.W, self.mu_head.gW), (self.mu_head.b, self.mu_head.gb),
                                 (self.lv_head.W, self.lv_head.gW), (self.lv_head.b, self.lv_head.gb)]
                + self.dec.pg())

    def p(self):
        return (self.enc.p() + [self.mu_head.W, self.mu_head.b, self.lv_head.W, self.lv_head.b]
                + self.dec.p())

    def decode(self, z):
        return sigmoid(self.dec.forward(z))

    def fit(self, X, beta=0.25, steps=4000, lr=2e-3, batch=128, seed=0):
        rng = np.random.default_rng(seed)
        st = [(np.zeros_like(p), np.zeros_like(p)) for p in self.p()]
        for t in range(1, steps + 1):
            xb = X[rng.choice(len(X), batch, replace=False)]
            self.forward(xb, rng)
            self.loss(xb, beta)
            self.backward(beta)
            for (p, g), (m, v) in zip(self.pg(), st):
                m *= 0.9; m += 0.1 * g
                v *= 0.999; v += 0.001 * g**2
                p -= lr * (m / (1 - 0.9**t)) / (np.sqrt(v / (1 - 0.999**t)) + 1e-8)
        return self

## 3. An instrument for 'can it generate?'

This is the part that took three attempts, and the failures are instructive enough to keep in the docstring.

In [ ]:
# The question "can this model generate?" needs an instrument. Reconstruction error
# cannot answer it, and neither can a human squinting at 8x8 blobs. Train a cheap
# classifier on the real digits and use it to label decoded samples.
judge = LogisticRegression(max_iter=2000).fit(X, y)
print("reference classifier accuracy on real digits:", round(judge.score(X, y), 4))


def digits_reachable(model, n=2000, seed=1):
    """Digits appearing as more than 1% of n decoded prior draws.

    Two weaker instruments were tried first. Nearest-neighbour distance to a
    training image separated a plain autoencoder from a VAE by 1.24 against 1.14,
    no story at all. Classifier *confidence* looked convincing on one seed and
    averaged to nothing over three, because a blurry blob is confidently labelled
    a 1. Coverage asks the right question: not "does this look like a digit" but
    "do these look like all the digits".
    """
    z = np.random.default_rng(seed).standard_normal((n, model.latent))
    hist = np.bincount(judge.predict(model.decode(z)), minlength=10) / n
    return int((hist > 0.01).sum())

## 4. What the KL term buys

Four models, four KL weights. Watch the reconstruction column stay flat across the first three while the last column does not.

In [ ]:
# Train one model per KL weight and keep them; every cell below reuses these.
models = {}
print(f"{'beta':>6}{'recon':>9}{'code spread':>14}{'digits reachable':>19}")
for beta in (0.0, 0.25, 1.0, 4.0):
    m = VAE(64, seed=0).fit(X, beta=beta)
    m.forward(X, np.random.default_rng(0), sample=False)
    recon, kl = m.loss(X, beta)
    models[beta] = m
    print(f"{beta:>6.2f}{recon:>9.2f}{m.mu.std():>14.2f}{digits_reachable(m):>13} of 10")

print("\nNear-identical reconstruction for the first three, and completely")
print("different answers to whether you can sample from them.")

## 5. Why, drawn

The circles mark one and two standard deviations of the prior. That region is the entire input domain of generation: draw `z ~ N(0, I)`, decode, done. Anything the encoder placed far outside it can never come back out.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (beta, m) in zip(axes, models.items()):
    m.forward(X, np.random.default_rng(0), sample=False)
    ax.scatter(m.mu[:, 0], m.mu[:, 1], c=y, cmap="tab10", s=5, alpha=0.75, linewidths=0)
    for r, ls in ((1, "-"), (2, "--")):           # the reach of the prior
        ax.add_patch(plt.Circle((0, 0), r, fill=False, color="black", ls=ls, lw=1.2))
    lim = max(3.0, np.abs(m.mu).max() * 1.05)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.set_title(f"beta = {beta}\n{digits_reachable(m)} of 10 digits from the prior")
    ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print("The circles are all that generation samples from. At beta = 0 the codes")
print("sprawl far outside them, so most digits are simply unreachable.")

## 6. The manifold the decoder learned

Two latent dimensions means the decoder can be inspected exhaustively. Nothing in the picture below is a training image.

In [ ]:
# With two latent dimensions the decoder can be inspected exhaustively.
from scipy.stats import norm

m = models[0.25]
n = 15
g = norm.ppf(np.linspace(0.02, 0.98, n))          # walk the prior's own quantiles
zz = np.array([[a, b] for b in g[::-1] for a in g])
tiles = m.decode(zz).reshape(n, n, 8, 8)

pad, step = 1, 9
canvas = np.zeros((n * step + pad, n * step + pad))
for i in range(n):
    for j in range(n):
        canvas[pad + i * step: pad + i * step + 8, pad + j * step: pad + j * step + 8] = tiles[i, j]

plt.figure(figsize=(6.5, 6.5))
plt.imshow(canvas, cmap="Greys", interpolation="nearest")
plt.xticks([]); plt.yticks([]); plt.grid(False)
plt.title("Every square is a digit the model invented")
plt.show()

## 7. The reparameterisation trick, ablated

Gradients do not flow through sampling. Writing `z = mu + sigma * eps` with `eps` drawn separately makes the randomness an input rather than an operation, and the derivatives become ordinary.

The cell below keeps the forward pass identical and cuts only the backward path, so the comparison isolates exactly the trick and nothing else.

In [ ]:
# Cut the gradient path through the sampling step and keep everything else the
# same. The forward pass is identical; only the backward differs.
class NoTrickVAE(VAE):
    def backward(self, beta):
        n = len(self.x)
        g_z = self.dec.backward((self.p_out - self.x) / n)
        # A naive sampling node gives the encoder nothing: z is treated as a
        # constant with respect to mu and sigma. Only the KL gradients survive.
        g_mu = beta * self.mu / n
        g_lv = beta * 0.5 * (np.exp(np.clip(self.lv, -20, 20)) - 1) / n
        self.enc.backward(self.mu_head.backward(g_mu) + self.lv_head.backward(g_lv))


for label, cls in (("with the trick", VAE), ("gradient path cut", NoTrickVAE)):
    m = cls(64, seed=0).fit(X, beta=1.0)
    m.forward(X, np.random.default_rng(0), sample=False)
    recon, _ = m.loss(X, 1.0)
    kl_dim = -0.5 * (1 + m.lv - m.mu**2 - np.exp(np.clip(m.lv, -20, 20))).mean(0)
    print(f"{label:>20}: recon {recon:6.2f}   active latent units {(kl_dim > 0.01).sum()}")

print("\nCutting the path does not degrade the model, it stops the encoder")
print("learning at all. The decoder falls back to emitting an average digit.")

## 8. Posterior collapse

Turn the KL weight up far enough and the encoder outputs the prior for every input, driving the KL term to exactly zero, while the decoder learns to emit an average digit. It looks like a bug; it is the optimum of the loss as written.

The diagnostic is **active units**: latent dimensions whose per-dimension KL clears a small threshold. When that count hits zero, no amount of further training helps.

In [ ]:
# Posterior collapse looks like a bug and is the optimum of the loss you wrote.
for beta in (1.0, 4.0):
    m = models[beta]
    m.forward(X, np.random.default_rng(0), sample=False)
    kl_dim = -0.5 * (1 + m.lv - m.mu**2 - np.exp(np.clip(m.lv, -20, 20))).mean(0)
    print(f"beta={beta}: per-dimension KL {np.round(kl_dim, 4)}  "
          f"active {(kl_dim > 0.01).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(9, 2.6))
for ax, beta in zip(axes, (1.0, 4.0)):
    z = np.random.default_rng(2).standard_normal((8, 2))
    ax.imshow(np.hstack(models[beta].decode(z).reshape(8, 8, 8)), cmap="Greys")
    ax.set_title(f"eight prior samples, beta = {beta}")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
plt.tight_layout(); plt.show()

## 9. Bernoulli or Gaussian

The likelihood is a statement about the data. Its practical consequence is bigger than it looks, because the two reconstruction terms carry different magnitudes, so the same `beta` weighs the KL differently against each.

In [ ]:
# The likelihood is a statement about the data, not a loss-function detail, and
# it changes what a given beta buys because the two terms carry different scales.
class GaussianVAE(VAE):
    def loss(self, x, beta):
        self.p_out, self.x = self.dec.forward(self.z), x      # no squashing
        recon = 0.5 * ((x - self.p_out) ** 2).sum(1)
        kl = -0.5 * (1 + self.lv - self.mu**2 - np.exp(np.clip(self.lv, -20, 20))).sum(1)
        return recon.mean(), kl.mean()

    def decode(self, z):
        return self.dec.forward(z)


print(f"{'likelihood':>12}{'beta':>7}{'pixel rmse':>13}{'code spread':>14}{'digits':>9}")
for name, cls in (("bernoulli", VAE), ("gaussian", GaussianVAE)):
    for beta in (0.25, 1.0):
        m = cls(64, seed=0).fit(X, beta=beta)
        m.forward(X, np.random.default_rng(0), sample=False)
        rmse = np.sqrt(np.mean((m.decode(m.mu) - X) ** 2))
        print(f"{name:>12}{beta:>7}{rmse:>13.4f}{m.mu.std():>14.2f}"
              f"{digits_reachable(m):>6} of 10")

print("\nThe same beta = 1 that is healthy under a Bernoulli likelihood collapses")
print("the posterior entirely under a Gaussian one.")

## Exercises

1. **Find the sweet spot.** Sweep `beta` from 0 to 2 in finer steps, plotting reconstruction and digits-reachable together. On this data both favour roughly 0.1 to 0.25, not the standard 1.0. Does that hold if you widen the hidden layer?
2. **KL annealing.** Ramp `beta` from 0 to 4 linearly over training instead of holding it fixed. Does the model still collapse? This is the standard fix from Bowman et al. (2016).
3. **More latent dimensions.** Go from 2 to 10. Count active units at each `beta`. You should find the model switches off the dimensions it does not need, which is partial collapse rather than total.
4. **Interpolate.** Encode two digits of different classes, walk a straight line between their codes, and decode each step. Repeat with the `beta = 0` model and compare what the path passes through.
5. **Free bits.** Clamp the per-dimension KL below a floor (`max(kl_j, lambda)`) so the optimiser cannot drive any single dimension to zero. Check whether it prevents collapse at `beta = 4`.


## Further reading

- Kingma & Welling (2014), [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114)
- Rezende, Mohamed & Wierstra (2014), [Stochastic Backpropagation and Approximate Inference in Deep Generative Models](https://arxiv.org/abs/1401.4082), the independent derivation
- Higgins et al. (2017), [beta-VAE](https://openreview.net/forum?id=Sy2fzU9gl)
- Bowman et al. (2016), [Generating Sentences from a Continuous Space](https://arxiv.org/abs/1511.06349), on posterior collapse and KL annealing
- [Approximate Inference: From MCMC to Variational Bayes](https://sesen.ai/blog/approximate-inference-mcmc-variational-bayes), the same bound without a neural network
